# Clasificador de textos en esperanto: Segunda Guerra Mundial, Guerra Civil Española y Guerras Napoleónicas

Minería de Textos · CUNEF · 2026/2027

## Fase 1 · Dataset

### Exploración de fuentes

**Objetivo.** Antes de construir el dataset comprobamos (30/10/2026) si Vikipedio (Wikipedia en esperanto) tiene texto suficiente para las tres clases. Para ello consultamos su API de MediaWiki en modo de solo lectura. La bitácora completa está en `docs/bitacora_exploracion_fase1.md` del repositorio.

**Por qué estas clases.** Empezamos con WWI, WWII y Guerra Fría, pero a profundidad 1 compartían 32 artículos (sobre todo en 1943–1949 y en el periodo de entreguerras). Eso mete ruido en las etiquetas y limita la precisión. Las sustituimos por tres guerras del mismo tema y con muy pocos artículos en común. Antes descartamos otras candidatas por falta de datos: Vietnam, Afganistán, Aceh, la guerra civil china y Corea.

**Hallazgos (páginas directas, recuento completo).**

| | Segunda Guerra Mundial | Guerra Civil Española | Guerras Napoleónicas |
|---|---|---|---|
| Categoría | `Kategorio:Dua Mondmilito` | `Kategorio:Hispana Enlanda Milito` | `Kategorio:Napoleonaj militoj` |
| Artículos directos | 132 | 54 | 41 |
| Subcategorías | 27 | 4 | 8 |
| Palabras mín / mediana / máx | 12 / 284 / 12.238 | 58 / 386 / 5.050 | 76 / 279 / 1.780 |
| Párrafos (≥10 palabras) por artículo | 25,1 | 14,9 | 7,3 |
| Párrafos en páginas directas | 3.312 | 802 | 298 |

**Solapes entre páginas directas:** ninguno.

**Valoración.** Es viable. Las clases son del mismo tema, como pide el enunciado. La Guerra Civil Española y la Segunda Guerra Mundial son cercanas (1936–1945, fascismo, intervención alemana e italiana), así que el problema no es trivial. Con solo las páginas directas, las Guerras Napoleónicas no llegan a 300 párrafos, por lo que estudiamos las subcategorías.

**Incidencias resueltas.** Los nombres de categoría distinguen mayúsculas. La API de extractos solo devuelve un texto completo por petición. Además, hubo que gestionar el error 429 (demasiadas peticiones) con reintentos y pausas.

### Estudio de profundidad de subcategorías

**Objetivo.** Medimos (30/10/2026) cuántos párrafos únicos se obtienen al bajar 0, 1 o 2 niveles en el árbol de categorías de cada clase. Se descargaron los 1.306 artículos distintos hasta profundidad 2, así que es un recuento completo. El detalle está en `docs/estudio_profundidad_fase1.md` del repositorio.

**Párrafos únicos por clase** (líneas de 10 palabras o más, sin contar los artículos que comparten varias clases):

| Profundidad | Segunda Guerra Mundial | Guerra Civil Española | Guerras Napoleónicas |
|---|---:|---:|---:|
| 0 (páginas directas) | 3.312 | 802 | 298 |
| 1 | 5.371 | 1.689 | 735 |
| 2 | 11.880 | 2.034 | 746 |

**Solapes.** A profundidad 1 solo hay 2 artículos compartidos, ambos entre la Segunda Guerra Mundial y la Guerra Civil Española: Hugo Sperrle e Hispana ekonomia miraklo. A profundidad 2 son 9. Las Guerras Napoleónicas no comparten ninguno.

**Decisiones.**
1. **Profundidad 1.** Es la mínima con la que las Guerras Napoleónicas superan los 300 párrafos (735). La profundidad 2 apenas les aporta (+11) y duplica la Segunda Guerra Mundial.
2. **Excluir:**
   - "Sekvoj de la Dua Mondmilito" y "Frankismo", porque son de posguerra y por ellas entra el solape "Hispana ekonomia miraklo".
   - "Brit-usona milito de 1812", porque es otra guerra.
   - El artículo "Listo de svedoj kiuj mortis en la Hispana Enlanda Milito", porque es una lista de nombres.

   Las subcategorías dudosas (biografías, lugares, obras) se mantienen y su ruido se mide más adelante.
3. **Descartar los artículos presentes en más de una clase.** Tras las exclusiones solo queda uno (Hugo Sperrle).
4. **Equilibrar las clases** recortando cada una al tamaño de la menor (Guerras Napoleónicas, unos 720 párrafos), con muestreo aleatorio y semilla fija. En la Fase 3 se compara con el dataset sin equilibrar.
5. **Conservar números y fechas** por ahora. En la Fase 3 se entrena con y sin ellos, porque los años (1799–1815, 1936–1939, 1939–1945) pueden delatar la clase casi solos.

### Generación del dataset

El código de generación está en el cuaderno, como pide el enunciado, pero se puede desactivar con `GENERAR_DATASET`. Con `False`, el cuaderno carga el CSV ya generado desde el repositorio. Con `True`, lo vuelve a construir desde la API de Vikipedio, lo que tarda varios minutos por las pausas entre peticiones.

In [68]:
# --- Importaciones ---
import re
import time
import random
from collections import deque, Counter

import requests
import pandas as pd

# --- Parámetros de ejecución ---
GENERAR_DATASET = False   # True: construir el dataset desde la API · False: cargar el CSV del repo

# --- Fuente y categorías (decisiones del estudio de profundidad) ---
API_URL = "https://eo.wikipedia.org/w/api.php"
CATEGORIAS = {                                   # clase -> categoría raíz (distingue mayúsculas)
    "Segunda Guerra Mundial": "Kategorio:Dua Mondmilito",
    "Guerra Civil Española": "Kategorio:Hispana Enlanda Milito",
    "Guerras Napoleónicas": "Kategorio:Napoleonaj militoj",
}
PROFUNDIDAD = 1                                  # decisión 1
SUBCATS_EXCLUIDAS = {                            # decisión 2
    "Kategorio:Sekvoj de la Dua Mondmilito",     # posguerra
    "Kategorio:Frankismo",                       # posguerra (dictadura 1939-1975)
    "Kategorio:Brit-usona milito de 1812",       # otra guerra
}
ARTICULOS_EXCLUIDOS = {                          # decisión 2
    "Listo de svedoj kiuj mortis en la Hispana Enlanda Milito",  # lista de nombres
}

# --- Limpieza y equilibrado ---
MIN_PALABRAS = 10                                # párrafo = línea con 10 palabras o más
SEMILLA = 42                                     # decisión 4 (reproducibilidad)

# --- Cortesía con la API (evita el error 429) ---
PAUSA = 1.0                                      # segundos entre peticiones
MAX_REINTENTOS = 4
USER_AGENT = "ClasificadorMilito/0.1 (https://github.com/Freibott/clasificador-milito; práctica CUNEF)"

# --- Rutas ---
RUTA_CSV = "data/dataset.csv"
URL_CSV_REMOTO = "https://raw.githubusercontent.com/Freibott/clasificador-milito/main/data/dataset.csv"

In [69]:
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})  # Wikimedia exige identificarse


def consultar_api(params):
    """Hace una petición GET a la API de MediaWiki y devuelve el JSON como diccionario.

    Centraliza la identificación (User-Agent), las pausas entre peticiones y los
    reintentos con espera creciente ante el límite de peticiones (error 429).
    """
    p = dict(params)                 # copia: no modificamos el diccionario de quien llama
    # Establecemos formato json
    p["format"] = "json"
    p["formatversion"] = 2           # MediaWiki: devuelve listas en vez de diccionarios por ID

    for intento in range(MAX_REINTENTOS):
        respuesta = session.get(API_URL, params=p, timeout=30)

        # 429 -> demasiadas peticiones, 5xx -> error del servidor: se reintenta
        if respuesta.status_code == 429 or respuesta.status_code >= 500:
            time.sleep(PAUSA * 2**intento)   # espera creciente (1, 2, 4, 8 s)
            continue

        respuesta.raise_for_status()         # otros errores HTTP (400, 403, 404): no se reintentan
        datos = respuesta.json()

        # MediaWiki responde 200 incluso con errores de la consulta
        if "error" in datos:
            raise RuntimeError(f"Error en la API de MediaWiki: {datos['error']['info']}")

        time.sleep(PAUSA)                    # pausa para no saturar el servidor
        return datos

    # Solo se llega aquí si se agotan todos los intentos
    raise RuntimeError(f"La API falló tras {MAX_REINTENTOS} intentos, último código: {respuesta.status_code}")

Comprobación: la raíz de la Segunda Guerra Mundial debe devolver 161 miembros directos (132 artículos, 27 subcategorías y 2 páginas de otros espacios de nombres).

In [70]:
if GENERAR_DATASET:
    datos = consultar_api({
        "action": "query",
        "list": "categorymembers",
        "cmtitle": CATEGORIAS["Segunda Guerra Mundial"],
        "cmlimit": 500,
    })
    len(datos["query"]["categorymembers"])  # → 161

### Bloque 2 · Artículos de cada clase

En este bloque se recopilan todos los artículos que componen cada una de las tres clases temáticas consultando la API de Wikipedia. Se utiliza una profundidad de 1 para obtener los artículos de la categoría raíz y los de sus subcategorías directas más relevantes. Este límite acota la búsqueda y evita el ruido temático o la dispersión excesiva que provocaría descender a niveles más profundos en la jerarquía.

In [71]:
if GENERAR_DATASET:
    def miembros_de_categoria(titulo, limite=500):
        """Devuelve (paginas, subcats): títulos de artículos (ns 0) y subcategorías (ns 14)."""
        params = {
            "action": "query",
            "list": "categorymembers",
            "cmtitle": titulo,
            "cmlimit": limite,
            "cmtype": "page|subcat",  # ignora ficheros e imágenes (ns 6)
        }
        paginas, subcats = [], []
        while True:
            datos = consultar_api(params)
            for m in datos["query"]["categorymembers"]:
                if m["ns"] == 0:
                    paginas.append(m["title"])
                elif m["ns"] == 14:
                    subcats.append(m["title"])
            if "continue" not in datos:
                break
            params.update(datos["continue"])
        return paginas, subcats

In [72]:
if GENERAR_DATASET:
    for lim in (500, 20):
        p, s = miembros_de_categoria(CATEGORIAS["Segunda Guerra Mundial"], limite=lim)
        print(lim, len(p), len(s), len(set(p)))

    for clase, raiz in CATEGORIAS.items():
        p, s = miembros_de_categoria(raiz)
        print(clase, len(p), len(s), s)

In [73]:
if GENERAR_DATASET:
    def recorrer_categoria(raiz, profundidad=PROFUNDIDAD, excluidas=SUBCATS_EXCLUIDAS):
        """Recorre el árbol de categorías en anchura (BFS) y devuelve {artículo: categoría_origen}."""
        cola = deque([(raiz, 0)])
        visitadas = {raiz}
        articulos = {}

        while cola:
            cat, nivel = cola.popleft()
            pags, subcats = miembros_de_categoria(cat)

            for pagina in pags:
                articulos.setdefault(pagina, cat)

            if nivel < profundidad:
                for s in subcats:
                    if s not in excluidas and s not in visitadas:
                        visitadas.add(s)
                        cola.append((s, nivel + 1))

        return articulos

In [74]:
if GENERAR_DATASET:
    sin_excl = {c: recorrer_categoria(r, excluidas=set()) for c, r in CATEGORIAS.items()}
    articulos_por_clase = {c: recorrer_categoria(r) for c, r in CATEGORIAS.items()}  # la usará el Bloque 3

    for c in CATEGORIAS:
        print(f"{c}: {len(sin_excl[c])} sin excl. · {len(articulos_por_clase[c])} con excl.")

    for nombre, d in (("sin", sin_excl), ("con", articulos_por_clase)):
        cuenta = Counter(t for arts in d.values() for t in arts)
        print(nombre, "solapes:", [t for t, n in cuenta.items() if n > 1])

### Bloque 3 · Descarga de texto

In [75]:
if GENERAR_DATASET:
    import os, json, random
    from tqdm.auto import tqdm

    def descargar_texto(titulo):
        params = {"action": "query", "prop": "extracts", "explaintext": 1,
                  "exsectionformat": "wiki", "redirects": 1, "titles": titulo,
                  "format": "json", "formatversion": 2}
        for _ in range(3):
            try:
                r = session.get(API_URL, params=params, timeout=30)
                r.raise_for_status()
                return r.json()["query"]["pages"][0].get("extract")  # None si falta
            except (requests.RequestException, ValueError, KeyError):
                time.sleep(2)
        return None

    t = descargar_texto("Napoleono Bonaparte")
    print(type(t), len(t), "==" in t)   # str, varios miles, True

In [76]:
if GENERAR_DATASET:
    titulos = sorted(set().union(*articulos_por_clase.values()) - ARTICULOS_EXCLUIDOS)
    print(len(titulos))   # → 553

In [77]:
if GENERAR_DATASET:
    RUTA_TEXTOS = "data/raw/textos.json"
    os.makedirs("data/raw", exist_ok=True)

    if os.path.exists(RUTA_TEXTOS):
        with open(RUTA_TEXTOS, encoding="utf-8") as f:
            textos = json.load(f)
    else:
        textos = {}
        for t in tqdm(titulos):
            textos[t] = descargar_texto(t)
            time.sleep(0.2)
        with open(RUTA_TEXTOS, "w", encoding="utf-8") as f:
            json.dump(textos, f, ensure_ascii=False)

In [78]:
if GENERAR_DATASET:
    print(len(textos))                                    # → 553
    nones = [t for t, v in textos.items() if v is None]
    cortos = [t for t, v in textos.items() if v is not None and len(v) < 200]
    print(len(nones), nones)                              # idealmente 0
    print(len(cortos), cortos)

    random.seed(42)
    for t in random.sample(sorted(textos), 2):
        print(t, "→", (textos[t] or "")[:300], "\n")

### Bloque 4 · Partir y limpiar párrafos

Un párrafo es una línea del texto plano con `MIN_PALABRAS` palabras o más, sin los títulos de sección (`== … ==`). Se limpian el HTML, los enlaces, las referencias y las pronunciaciones entre corchetes, y se sustituye `;` por `,` porque es el separador del CSV. Se descartan las citas bibliográficas (ISBN/ISSN o *Autor (año).*), que no son prosa.

In [79]:
if GENERAR_DATASET:
    import re

    # citas bibliográficas (ISBN/ISSN o "Autor (2011)." / "Autor (1860),") y avisos de traducción: no son prosa
    REFERENCIA = re.compile(r"\b(ISBN|ISSN)\b|\(\d{4}\)[.,:]|\btraduko de teksto\b")

    def limpiar(linea):
        linea = re.sub(r"<[^>]+>", "", linea)    # etiquetas HTML
        linea = re.sub(r"https?://\S+", "", linea)   # enlaces sueltos
        linea = re.sub(r"\[[^\]]*\]", "", linea)     # referencias [1], pronunciaciones [kanAÚ], [jul.]
        linea = linea.replace(";", ",")          # ';' es el separador del CSV
        return re.sub(r"\s+", " ", linea).strip()

    def partir_parrafos(texto):
        parrafos = []
        for linea in texto.split("\n"):
            linea = linea.strip()
            if not linea or linea.startswith("=="):   # líneas vacías y títulos de sección
                continue
            linea = limpiar(linea)
            if REFERENCIA.search(linea):
                continue
            if len(linea.split()) >= MIN_PALABRAS:    # párrafo = MIN_PALABRAS palabras o más
                parrafos.append(linea)
        return parrafos

    parrafos_por_articulo = {t: partir_parrafos(x) for t, x in textos.items() if x is not None}

    clases_de = {}   # título -> conjunto de clases en las que aparece
    for clase, arts in articulos_por_clase.items():
        for t in arts:
            clases_de.setdefault(t, set()).add(clase)

    for clase in CATEGORIAS:
        arts = [t for t in parrafos_por_articulo if clase in clases_de.get(t, ())]
        todos = [p for t in arts for p in parrafos_por_articulo[t]]
        print(f"{clase}: {len(arts)} artículos · {len(todos)} párrafos · "
              f"{len(set(todos))} únicos · {len(todos) / len(arts):.1f} por artículo")

    todos_p = [p for ps in parrafos_por_articulo.values() for p in ps]
    print("Párrafos con restos de HTML, enlaces o corchetes:",
          sum(bool(re.search(r"<|http|\[|\]", p)) for p in todos_p))

### Bloque 5 · Quitar compartidos y equilibrar

Se descartan los artículos que están en más de una clase (solo `Hugo Sperrle`) y los párrafos repetidos, dentro de una clase o entre clases. Después se recorta cada clase al tamaño de la menor con un muestreo aleatorio de semilla fija (`SEMILLA`).

In [80]:
if GENERAR_DATASET:
    # 1) descartar los artículos que están en más de una clase
    compartidos = sorted(t for t, cs in clases_de.items() if len(cs) > 1)
    print("Artículos compartidos descartados:", compartidos)

    por_clase = {c: [] for c in CATEGORIAS}
    for t, ps in parrafos_por_articulo.items():
        if t in compartidos or t not in clases_de:
            continue
        por_clase[next(iter(clases_de[t]))].extend(ps)

    # 2) quitar repetidos dentro de cada clase (conservando el orden)
    por_clase = {c: list(dict.fromkeys(ps)) for c, ps in por_clase.items()}

    # 3) quitar los párrafos idénticos que aparecen en más de una clase
    cuenta = Counter(p for ps in por_clase.values() for p in ps)
    repetidos_entre_clases = sum(1 for n in cuenta.values() if n > 1)
    por_clase = {c: [p for p in ps if cuenta[p] == 1] for c, ps in por_clase.items()}
    print("Párrafos idénticos en varias clases descartados:", repetidos_entre_clases)
    print("Antes de equilibrar:", {c: len(ps) for c, ps in por_clase.items()})

    # 4) equilibrar al tamaño de la clase menor (muestreo aleatorio con SEMILLA)
    n_min = min(len(ps) for ps in por_clase.values())
    rng = random.Random(SEMILLA)
    equilibrado = {c: rng.sample(ps, n_min) for c, ps in por_clase.items()}
    print("Después de equilibrar:", {c: len(ps) for c, ps in equilibrado.items()}, "· total", 3 * n_min)

### Bloque 6 · Exportar el dataset

Se mezcla el dataset equilibrado, se guarda como `data/dataset.csv` (UTF-8, separador `;`, columnas `text;class`) y se valida el contrato. Al final se muestran algunos párrafos al azar para medir el ruido a mano.

In [81]:
if GENERAR_DATASET:
    import pandas as pd

    filas = [(p, c) for c, ps in equilibrado.items() for p in ps]
    df = pd.DataFrame(filas, columns=["text", "class"])
    df = df.sample(frac=1, random_state=SEMILLA).reset_index(drop=True)   # mezclar las filas

    os.makedirs(os.path.dirname(RUTA_CSV), exist_ok=True)
    df.to_csv(RUTA_CSV, sep=";", index=False, encoding="utf-8")

    # validación del contrato: UTF-8, separador ';', columnas text;class
    chk = pd.read_csv(RUTA_CSV, sep=";", encoding="utf-8")
    assert list(chk.columns) == ["text", "class"]
    assert chk["text"].notna().all() and (chk["text"].str.strip().str.len() > 0).all()
    assert set(chk["class"]) == set(CATEGORIAS)
    assert chk["class"].value_counts().nunique() == 1          # clases equilibradas
    assert not chk["text"].str.contains(";").any()             # ningún ';' dentro del texto
    assert not chk["text"].duplicated().any()                  # sin párrafos repetidos
    assert (chk["text"].str.split().str.len() >= MIN_PALABRAS).all()   # todos con MIN_PALABRAS palabras o más
    print(chk.shape)
    print(chk["class"].value_counts())

    # muestra para medir el ruido a mano
    for c in CATEGORIAS:
        print("\n" + c)
        for p in chk[chk["class"] == c]["text"].sample(6, random_state=7):
            print("  -", p[:170])

### Bloque 7 · Cargar o generar el dataset

Según `GENERAR_DATASET`, el dataset se construye con los bloques anteriores (`True`) o se descarga del repositorio (`False`). En los dos casos se lee de un CSV con las columnas `text;class`.

In [82]:
# Con GENERAR_DATASET = True el CSV lo acaba de escribir el Bloque 6; con False se descarga del repo.
if GENERAR_DATASET:
    df = pd.read_csv(RUTA_CSV, sep=";", encoding="utf-8")
else:
    df = pd.read_csv(URL_CSV_REMOTO, sep=";", encoding="utf-8")

assert list(df.columns) == ["text", "class"] and set(df["class"]) == set(CATEGORIAS)
print(df.shape)
print(df["class"].value_counts())
df.head()

(1803, 2)
class
Guerra Civil Española     601
Guerras Napoleónicas      601
Segunda Guerra Mundial    601
Name: count, dtype: int64


,text,class
0,"Flago kun ruĝaj kaj nigraj vertikalaj strioj, ...",Guerra Civil Española
1,"Antonio MACHADO Ruiz (naskiĝis en Sevilo, la 2...",Guerra Civil Española
2,"De tiam, la ribelo eksplodis laŭlonge de la tu...",Guerras Napoleónicas
3,"Subleŭtenanto Josef BURŠÍK (generalmajoro), ko...",Segunda Guerra Mundial
4,Ĉirkaŭ la 7-a vespere la 23-an de februaro 194...,Segunda Guerra Mundial


## Fase 2 · NLP

## Fase 3 · Clasificación